In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    window,
    min,
    max,
    sum,
    count,
    min_by,
    max_by,
    round as spark_round,
    year,
    month,
    dayofmonth,
)


SILVER_PATH = (
    "s3a://kafka-spark-stock-project-kris/"
    "stock-market/silver/simulated/trades/"
)

GOLD_PATH = (
    "s3a://kafka-spark-stock-project-kris/"
    "stock-market/gold/simulated/ohlcv_1min/"
)


def main():

    spark = (
        SparkSession.builder
        .appName("StockMarketGoldOHLCVWriter")
        .config("spark.sql.session.timeZone", "UTC")
        .config(
            "spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.auth.IAMInstanceCredentialsProvider",
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    # -----------------------------------------------------
    # Read Silver
    # -----------------------------------------------------

    silver_df = (
        spark.read
        .parquet(SILVER_PATH)
    )

    print(f"Silver trades: {silver_df.count()}")

    # -----------------------------------------------------
    # Build 1-minute Gold candles
    # -----------------------------------------------------

    gold_df = (
        silver_df

        .groupBy(
            "symbol",
            window(
                col("event_timestamp"),
                "1 minute"
            ).alias("trade_window")
        )

        .agg(
            min_by(
                col("price"),
                col("event_timestamp")
            ).alias("open"),

            max("price").alias("high"),

            min("price").alias("low"),

            max_by(
                col("price"),
                col("event_timestamp")
            ).alias("close"),

            sum("volume").alias("volume"),

            count("*").alias("trade_count"),

            spark_round(
                sum(col("price") * col("volume"))
                / sum("volume"),
                4
            ).alias("vwap"),
        )

        .select(
            "symbol",

            col("trade_window.start")
            .alias("window_start"),

            col("trade_window.end")
            .alias("window_end"),

            "open",
            "high",
            "low",
            "close",
            "volume",
            "trade_count",
            "vwap",
        )

        # Gold partitions based on candle time
        .withColumn(
            "year",
            year(col("window_start"))
        )

        .withColumn(
            "month",
            month(col("window_start"))
        )

        .withColumn(
            "day",
            dayofmonth(col("window_start"))
        )
    )

    print(f"Gold candles: {gold_df.count()}")

    # -----------------------------------------------------
    # Write Gold
    # -----------------------------------------------------

    (
        gold_df.write
        .mode("overwrite")
        .option("compression", "snappy")
        .partitionBy(
            "year",
            "month",
            "day"
        )
        .parquet(GOLD_PATH)
    )

    print("=" * 60)
    print("Gold OHLCV write completed")
    print(f"Gold path: {GOLD_PATH}")
    print("=" * 60)

    spark.stop()


if __name__ == "__main__":
    main()